# 2026 호르무즈 해협 영어 해외언론 텍스트마이닝 — 재구성 버전

**연구 기간:** 2026-01-01 ~ 2026-08-31  
**핵심 키워드:** `Strait of Hormuz`, `Hormuz Strait`  
**원본 데이터:** 기존에 수집한 `01_newscatcher_raw.csv`

## 이 노트북의 원칙

이 버전은 기존 노트북의 **NewsCatcher 재수집 및 Diffbot 전체 수집을 제거**하고, 이미 확보한 원본 CSV에서 연구용 데이터셋을 다시 구성한다.

핵심 변경점:

1. NewsCatcher API를 다시 호출하지 않는다.
2. URL 중복만 먼저 정리하고, 원본 NewsCatcher 메타데이터를 최대한 보존한다.
3. `description`과 NewsCatcher의 잘린 `content`를 서로 다른 컬럼으로 보존한다.
4. 동일 제목을 발견해도 기사를 삭제하지 않고 **중복 그룹으로 표시**한다.
5. `hormuz_relevance_score`는 삭제 기준이 아니라 분석용 메타데이터/추출 우선순위로 사용한다.
6. 본문 추출은 이 노트북에서 42만 건 전체를 바로 실행하지 않는다.
7. 먼저 층화 표본으로 **Fundus / news-please / trafilatura** 등의 추출 성능을 비교한 뒤 기본 추출기를 결정한다.
8. 본문 확보 후 언어, 본문 품질, 신디케이션/중복 여부를 판단한다.

> `01_newscatcher_raw.csv`는 원본이므로 이 노트북에서 수정하거나 덮어쓰지 않는다.


## 1. 패키지와 프로젝트 경로

이 노트북은 기존 NewsCatcher 결과를 읽는 것이 목적이므로 API 수집 패키지는 사용하지 않는다.

처음 실행할 때 필요한 최소 패키지만 설치한다.


In [ ]:
%pip install -q pandas numpy requests python-dotenv tqdm langdetect scikit-learn


## 2. 기본 설정 및 데이터 위치 확인

노트북의 현재 작업 폴더가 `Hormuz_Research`이면 자동으로 찾는다.  
VS Code에서 다른 폴더를 작업 디렉터리로 잡은 경우에는 상위 폴더를 탐색한다.

**중요:** 아래 셀에서 찾는 파일은 기존 `01_newscatcher_raw.csv` 하나다.


In [ ]:
from pathlib import Path
import hashlib
import re
from urllib.parse import urlsplit, urlunsplit

import numpy as np
import pandas as pd

PROJECT_NAME = "Hormuz_Research"
RAW_FILENAME = "01_newscatcher_raw.csv"

START_DATE = "2026-01-01"
END_DATE = "2026-08-31"

# 현재 위치부터 상위 폴더까지 탐색
candidate_roots = [Path.cwd()] + list(Path.cwd().parents)

PROJECT_ROOT = None
for root in candidate_roots:
    if root.name == PROJECT_NAME and (root / "hormuz_2026_data" / RAW_FILENAME).exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    # 현재 폴더 자체가 프로젝트 폴더인 경우
    if (Path.cwd() / "hormuz_2026_data" / RAW_FILENAME).exists():
        PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Hormuz_Research/hormuz_2026_data/01_newscatcher_raw.csv를 찾지 못했습니다.\n"
        "VS Code에서 Hormuz_Research 폴더를 열고 다시 실행하세요."
    )

DATA_DIR = PROJECT_ROOT / "hormuz_2026_data"
RAW_NEWS_FILE = DATA_DIR / RAW_FILENAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RAW_NEWS_FILE:", RAW_NEWS_FILE)


## 3. 원본 NewsCatcher CSV 로드

이 단계에서는 원본 레코드를 삭제하지 않는다.

원본에 존재하는 주요 메타데이터를 `nc_` 접두사로 명확하게 보존한다. 특히 기존 노트북에서 버려졌던 다음 항목을 유지한다.

- `word_count`
- `paid_content`
- `country`
- `is_opinion`
- `rank`
- `duplicate_count`
- NewsCatcher 원본 `id`
- `published_date_precision`
- `description`
- `content`
- `is_content_truncated`

또한 `content`는 기사 본문으로 취급하지 않는다. NewsCatcher가 반환한 짧은 콘텐츠/스니펫으로만 보존한다.


In [ ]:
raw_df = pd.read_csv(
    RAW_NEWS_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

print("=" * 70)
print("RAW DATA")
print("=" * 70)
print(f"원본 레코드 수: {len(raw_df):,}")
print(f"컬럼 수: {len(raw_df.columns)}")
print()
print(raw_df.columns.tolist())

required_raw = ["title", "link", "published_date", "language", "description", "content"]
missing_raw = [c for c in required_raw if c not in raw_df.columns]

if missing_raw:
    raise KeyError(f"원본 CSV에 필요한 컬럼이 없습니다: {missing_raw}")

print()
print("필수 원본 컬럼 확인 완료")


## 4. URL 정규화

URL 정규화는 **같은 URL의 기술적 변형을 비교하기 위한 단계**다.

이번 버전에서는 원래 URL의 query string을 임의로 제거하지 않는다.  
따라서 URL 정규화만으로 과도하게 서로 다른 URL을 합치지 않는다.

- scheme 소문자화
- hostname 소문자화
- fragment 제거
- 마지막 `/` 제거

URL 중복은 이후 하나의 대표 행으로 만들되, 원본 NewsCatcher ID와 중복 개수를 별도로 보존한다.


In [ ]:
def normalize_url(url):
    if pd.isna(url):
        return None

    url = str(url).strip()
    if not url:
        return None

    try:
        parsed = urlsplit(url)

        if parsed.scheme.lower() not in {"http", "https"}:
            return None

        if not parsed.netloc:
            return None

        return urlunsplit((
            parsed.scheme.lower(),
            parsed.netloc.lower(),
            parsed.path.rstrip("/") or "/",
            parsed.query,
            "",
        ))
    except Exception:
        return None


raw_df["url_raw"] = raw_df["link"].astype("string")
raw_df["url"] = raw_df["url_raw"].apply(normalize_url)

print("유효 URL:", raw_df["url"].notna().sum())
print("유효 URL 없음:", raw_df["url"].isna().sum())


## 5. URL 기준 기사 인덱스 재구성

여기서 **URL 중복만 제거**한다.

같은 URL이 여러 NewsCatcher 레코드에 존재할 경우:

- 원본 NewsCatcher ID들을 `nc_ids`에 보존
- 원본 레코드 수를 `nc_duplicate_count`에 기록
- 서로 다른 제목이 있었는지 `nc_multiple_titles`로 기록
- 대표 행은 메타데이터가 더 많이 채워진 행을 우선하고, 동률이면 최초 수집/발행 정보를 고려해 결정

이 단계는 제목 중복 제거와 다르다. 동일 제목을 가진 서로 다른 URL은 삭제하지 않는다.


In [ ]:
# 원본 필드 → 연구용 필드
RENAME_MAP = {
    "id": "nc_id",
    "title": "title",
    "author": "nc_author",
    "authors": "nc_authors",
    "journalists": "nc_journalists",
    "published_date": "published_date",
    "published_date_precision": "nc_published_date_precision",
    "updated_date": "nc_updated_date",
    "updated_date_precision": "nc_updated_date_precision",
    "description": "nc_description",
    "content": "nc_content_snippet",
    "word_count": "nc_word_count",
    "is_opinion": "nc_is_opinion",
    "paid_content": "nc_paid_content",
    "country": "nc_country",
    "rights": "nc_rights",
    "rank": "nc_rank",
    "media": "nc_media",
    "language": "nc_language",
    "name_source": "nc_source_name",
    "domain_url": "nc_domain_url",
    "full_domain_url": "nc_full_domain_url",
    "parent_url": "nc_parent_url",
    "is_headline": "nc_is_headline",
    "all_links": "nc_all_links",
    "all_domain_links": "nc_all_domain_links",
    "score": "nc_score",
    "duplicate_count": "nc_duplicate_count_vendor",
    "duplicate_articles_group_id": "nc_duplicate_group_id_vendor",
    "is_content_truncated": "nc_is_content_truncated",
    "link": "nc_original_link",
}

work = raw_df.copy()

for raw_col, new_col in RENAME_MAP.items():
    if raw_col in work.columns:
        work[new_col] = work[raw_col]

# 메타데이터 완성도: 대표 URL 행을 결정할 때만 사용
metadata_priority_cols = [
    "title", "nc_description", "nc_content_snippet", "published_date",
    "nc_language", "nc_source_name", "nc_country", "nc_word_count",
]

work["_metadata_completeness"] = work[metadata_priority_cols].notna().sum(axis=1)

# 날짜는 정렬용으로만 사용. 최종 연구 날짜의 기준은 NewsCatcher published_date.
work["_published_sort"] = pd.to_datetime(
    work["published_date"], errors="coerce", utc=True
)

# 대표 행을 결정하기 위한 안정적인 정렬
work = work.sort_values(
    by=["url", "_metadata_completeness", "_published_sort"],
    ascending=[True, False, True],
    kind="mergesort",
)

# URL별 원본 ID/제목/도메인 정보 보존
url_groups = work.groupby("url", dropna=True)

url_meta = url_groups.agg(
    nc_ids=("nc_id", lambda s: " | ".join(pd.Series(s.dropna().astype(str)).drop_duplicates())),
    nc_url_record_count=("url", "size"),
    nc_multiple_titles=("title", lambda s: s.dropna().astype(str).nunique() > 1),
    nc_title_count=("title", lambda s: s.dropna().astype(str).nunique()),
    nc_source_count=("nc_source_name", lambda s: s.dropna().astype(str).nunique()),
).reset_index()

# URL별 대표 행 1개
article_index = (
    work.drop_duplicates(subset=["url"], keep="first")
        .drop(columns=["_metadata_completeness", "_published_sort"], errors="ignore")
        .merge(url_meta, on="url", how="left")
        .reset_index(drop=True)
)

# 연구용 article_id는 정규화 URL 기반으로 생성
article_index["article_id"] = article_index["url"].apply(
    lambda u: hashlib.sha1(str(u).encode("utf-8")).hexdigest()[:16]
)

print("=" * 70)
print("URL DEDUPLICATION")
print("=" * 70)
print(f"원본 레코드: {len(raw_df):,}")
print(f"유효 URL 레코드: {raw_df['url'].notna().sum():,}")
print(f"URL 중복 제거 후 기사: {len(article_index):,}")
print(f"URL 중복으로 합쳐진 레코드: {raw_df['url'].notna().sum() - len(article_index):,}")
print(f"동일 URL에서 제목이 2개 이상인 그룹: {article_index['nc_multiple_titles'].sum():,}")


## 6. 기본 메타데이터와 연구 기간 플래그

이 단계에서는 기존처럼 제목 길이, 영어 여부 등을 이유로 기사를 대량 삭제하지 않는다.

특히 다음은 **플래그**로만 관리한다.

- 제목이 지나치게 짧음
- 제목이 지나치게 김
- URL이 동영상/라이브/태그 페이지처럼 보임
- 알려진 aggregator/republication 도메인
- NewsCatcher 관련성 점수 0/1/2

연구 기간 밖의 기사와 URL이 없는 원본 레코드는 연구 대상 후보에서 제외할 수 있지만, 원본 CSV에는 영향을 주지 않는다.


In [ ]:
def get_domain(url):
    try:
        return urlsplit(str(url)).netloc.lower().split(":")[0]
    except Exception:
        return ""


def normalize_title(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def hormuz_relevance(title, description, snippet):
    text = " ".join([
        str(title) if pd.notna(title) else "",
        str(description) if pd.notna(description) else "",
        str(snippet) if pd.notna(snippet) else "",
    ]).lower()

    if "strait of hormuz" in text or "hormuz strait" in text:
        return 2
    if "hormuz" in text:
        return 1
    return 0


article_index["domain"] = article_index["url"].apply(get_domain)

article_index["normalized_title"] = article_index["title"].apply(normalize_title)

article_index["title_char_count"] = (
    article_index["title"].fillna("").astype(str).str.len()
)

article_index["flag_title_short"] = article_index["title_char_count"] < 10
article_index["flag_title_very_long"] = article_index["title_char_count"] > 300

article_index["flag_url_video"] = article_index["url"].str.lower().str.contains(
    r"/video[s]?/|/watch/|/live/", regex=True, na=False
)

article_index["flag_url_non_article"] = article_index["url"].str.lower().str.contains(
    r"/tag/|/tags/|/category/|/author/|/search/", regex=True, na=False
)

AGGREGATOR_REVIEW_DOMAINS = {
    "wn.com",
    "headtopics.com",
    "newsbreak.com",
    "msn.com",
    "yahoo.com",
    "finance.yahoo.com",
    "investing.com",
    "tradingview.com",
    "dailyhunt.in",
}

article_index["flag_aggregator_review"] = article_index["domain"].apply(
    lambda d: (
        d in AGGREGATOR_REVIEW_DOMAINS
        or any(d.endswith("." + x) for x in AGGREGATOR_REVIEW_DOMAINS)
    )
)

article_index["hormuz_relevance_score"] = [
    hormuz_relevance(t, d, s)
    for t, d, s in zip(
        article_index["title"],
        article_index["nc_description"],
        article_index["nc_content_snippet"],
    )
]

published_dt = pd.to_datetime(
    article_index["published_date"],
    errors="coerce",
    utc=True,
)

start_dt = pd.Timestamp(START_DATE, tz="UTC")
end_dt_exclusive = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1)

article_index["in_research_period"] = (
    published_dt >= start_dt
) & (
    published_dt < end_dt_exclusive
)

article_index["flag_missing_title"] = (
    article_index["title"].fillna("").astype(str).str.strip().eq("")
)

article_index["flag_missing_description"] = article_index["nc_description"].isna()
article_index["flag_paid_content"] = (
    article_index["nc_paid_content"].fillna(False).astype(str).str.lower().isin(
        {"true", "1", "yes"}
    )
)

print("기사 수:", len(article_index))
print("연구 기간 내 기사:", article_index["in_research_period"].sum())
print("연구 기간 밖/날짜 미상:", (~article_index["in_research_period"]).sum())


## 7. 동일 제목은 삭제하지 않고 그룹화

기존 노트북은 `normalized_title.duplicated(keep="first")` 방식으로 제목이 같은 기사를 삭제했다.

이 버전에서는 그렇게 하지 않는다.

대신 다음 정보를 만든다.

- `title_dup_group_id`
- `title_dup_group_size`
- `title_dup_n_domains`
- `title_dup_flag`

따라서 서로 다른 언론사의 같은 제목, 같은 기사의 재게시, 실제 중복 가능성을 나중에 본문 확보 후 판단할 수 있다.


In [ ]:
valid_title_mask = article_index["normalized_title"].ne("")

article_index["title_dup_group_id"] = pd.NA
article_index.loc[valid_title_mask, "title_dup_group_id"] = (
    article_index.loc[valid_title_mask, "normalized_title"]
    .apply(lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()[:16])
)

title_group_stats = (
    article_index.loc[valid_title_mask]
    .groupby("title_dup_group_id")
    .agg(
        title_dup_group_size=("article_id", "size"),
        title_dup_n_domains=("domain", "nunique"),
    )
    .reset_index()
)

article_index = article_index.merge(
    title_group_stats,
    on="title_dup_group_id",
    how="left",
)

article_index["title_dup_group_size"] = (
    article_index["title_dup_group_size"].fillna(1).astype(int)
)

article_index["title_dup_n_domains"] = (
    article_index["title_dup_n_domains"].fillna(1).astype(int)
)

article_index["title_dup_flag"] = article_index["title_dup_group_size"] > 1

print("=" * 70)
print("TITLE DUPLICATE GROUPING")
print("=" * 70)
print("제목 중복 그룹에 포함된 기사:", article_index["title_dup_flag"].sum())
print("중복 제목 그룹 수:", title_group_stats["title_dup_group_id"].nunique())

print()
display(
    article_index.loc[
        article_index["title_dup_flag"],
        [
            "article_id", "title", "domain",
            "title_dup_group_size", "title_dup_n_domains"
        ]
    ].head(20)
)


## 8. 영어/비영어 판정은 본문 확보 후 수행

NewsCatcher의 `language=en`은 유용한 메타데이터지만 기사 본문 전체가 영어라는 것을 확정하는 기준으로 사용하지 않는다.

따라서 여기서는 원본 언어값을 그대로 보존하고, 본문 추출 후 `language_detected`를 별도로 계산한다.

아래 `langdetect` 테스트 함수는 최종 삭제가 아니라 **본문 품질 진단용**이다.


In [ ]:
from langdetect import detect, LangDetectException


def detect_language_safe(text):
    text = str(text or "").strip()

    if len(text) < 50:
        return None

    try:
        return detect(text)
    except LangDetectException:
        return None
    except Exception:
        return None


print("NewsCatcher language 분포:")
display(
    article_index["nc_language"]
    .fillna("(missing)")
    .astype(str)
    .str.lower()
    .value_counts()
    .head(20)
)


## 9. 메타데이터 진단

이 셀은 기사를 삭제하지 않는다.

확인할 항목:

- 도메인 집중도
- 국가 분포
- 월별 분포
- NewsCatcher 관련성 점수
- `paid_content`
- NewsCatcher `word_count`
- 제목 중복 그룹
- aggregator/republication 검토 플래그

특정 도메인의 기사 수가 많다는 사실만으로 편향이라고 판단하지 않는다. 이후 본문 추출 성공률과 실제 기사 유형을 함께 확인한다.


In [ ]:
diagnostic = article_index[
    article_index["in_research_period"]
].copy()

diagnostic["published_dt"] = pd.to_datetime(
    diagnostic["published_date"], errors="coerce", utc=True
)

diagnostic["month"] = diagnostic["published_dt"].dt.to_period("M").astype(str)

print("=" * 70)
print("METADATA DIAGNOSTIC")
print("=" * 70)

print(f"연구 기간 내 URL 중복 제거 기사: {len(diagnostic):,}")
print(f"유일 도메인 수: {diagnostic['domain'].nunique():,}")

print()
print("상위 30개 도메인:")
display(
    diagnostic["domain"]
    .value_counts()
    .head(30)
    .rename_axis("domain")
    .reset_index(name="article_count")
)

print()
print("상위 도메인 집중도:")
domain_counts = diagnostic["domain"].value_counts()
for n in [10, 20, 50, 100]:
    top_n = domain_counts.head(n).sum()
    ratio = top_n / len(diagnostic) * 100
    print(f"상위 {n:>3}개: {top_n:>8,}건 ({ratio:6.2f}%)")

print()
print("호르무즈 관련성 점수:")
display(
    diagnostic["hormuz_relevance_score"]
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .reset_index(name="article_count")
)

print()
print("월별 기사 수:")
display(
    diagnostic["month"]
    .value_counts()
    .sort_index()
    .rename_axis("month")
    .reset_index(name="article_count")
)

print()
print("국가 분포 상위:")
display(
    diagnostic["nc_country"]
    .fillna("(unknown)")
    .astype(str)
    .value_counts()
    .head(20)
    .rename_axis("country")
    .reset_index(name="article_count")
)

print()
print("NewsCatcher word_count 요약:")
display(
    pd.to_numeric(diagnostic["nc_word_count"], errors="coerce")
    .describe()
    .to_frame("nc_word_count")
)

print()
print("paid_content:")
display(
    diagnostic["flag_paid_content"]
    .value_counts()
    .rename_axis("paid_content")
    .reset_index(name="article_count")
)


## 10. 본문 추출 후보 만들기

본문 추출 후보는 **제목 중복으로 줄이지 않는다.**

기본 후보 조건은 연구 기간 안의 기사이고 URL이 정상인 경우다.  
`hormuz_relevance_score`, aggregator 플래그, `nc_word_count`, `paid_content` 등은 추출 우선순위와 사후 분석에 활용한다.

즉, 점수 0이라고 삭제하지 않는다.


In [ ]:
def valid_http_url(url):
    try:
        p = urlsplit(str(url))
        return p.scheme in {"http", "https"} and bool(p.netloc)
    except Exception:
        return False


article_index["valid_http_url"] = article_index["url"].apply(valid_http_url)

extraction_candidates = article_index[
    article_index["in_research_period"] &
    article_index["valid_http_url"]
].copy().reset_index(drop=True)

# 추출 우선순위용 플래그
extraction_candidates["extraction_priority"] = (
    extraction_candidates["hormuz_relevance_score"]
    .map({2: 0, 1: 1, 0: 2})
    .fillna(2)
)

print("=" * 70)
print("EXTRACTION CANDIDATES")
print("=" * 70)
print(f"본문 추출 후보: {len(extraction_candidates):,}")
print("제목 중복 때문에 삭제된 기사: 0")

display(
    extraction_candidates[
        [
            "article_id", "title", "domain",
            "hormuz_relevance_score",
            "flag_aggregator_review",
            "nc_word_count",
            "nc_paid_content",
        ]
    ].head(10)
)


## 11. 추출기 벤치마크용 층화 표본

전체 수십만 건에 추출기를 먼저 돌리지 않는다.

추출기와 도메인이 강하게 얽히는 문제를 줄이기 위해 다음 축을 사용해 표본을 만든다.

- `hormuz_relevance_score`
- `flag_aggregator_review`
- `nc_country`
- 발행 월

각 층에서 최대 `PER_STRATUM`건을 고정 seed로 추출한다.

이 표본을 Fundus / news-please / trafilatura에 동일하게 적용하고, 일부는 사람이 직접 본문 품질을 확인한 뒤 기본 추출기와 fallback을 결정한다.


In [ ]:
RANDOM_SEED = 2026
PER_STRATUM = 10
BENCHMARK_SAMPLE_SIZE_LIMIT = 300

benchmark_pool = extraction_candidates.copy()

benchmark_pool["published_dt"] = pd.to_datetime(
    benchmark_pool["published_date"], errors="coerce", utc=True
)
benchmark_pool["month"] = benchmark_pool["published_dt"].dt.to_period("M").astype(str)

benchmark_pool["country_stratum"] = (
    benchmark_pool["nc_country"]
    .fillna("(unknown)")
    .astype(str)
)

benchmark_pool["source_type_stratum"] = np.where(
    benchmark_pool["flag_aggregator_review"],
    "aggregator_review",
    "other"
)

strata_cols = [
    "hormuz_relevance_score",
    "source_type_stratum",
    "country_stratum",
    "month",
]

parts = []

for _, group in benchmark_pool.groupby(strata_cols, dropna=False):
    parts.append(
        group.sample(
            n=min(PER_STRATUM, len(group)),
            random_state=RANDOM_SEED,
        )
    )

benchmark_sample = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates("article_id")
)

if len(benchmark_sample) > BENCHMARK_SAMPLE_SIZE_LIMIT:
    benchmark_sample = benchmark_sample.sample(
        n=BENCHMARK_SAMPLE_SIZE_LIMIT,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

print("=" * 70)
print("EXTRACTOR BENCHMARK SAMPLE")
print("=" * 70)
print(f"표본 기사 수: {len(benchmark_sample):,}")

display(
    benchmark_sample[
        [
            "article_id", "title", "domain",
            "nc_country", "month",
            "hormuz_relevance_score",
            "flag_aggregator_review",
            "url",
        ]
    ].head(30)
)


## 12. 본문 추출 결과 스키마

아직 여기서 전체 본문 추출을 실행하지 않는다.

추출 결과는 최소한 아래 정보를 남긴다.

- `article_id`
- `url`
- `extractor_used`
- `extraction_status`
- `failure_reason`
- `body`
- `body_length`
- `http_status`
- `final_url`
- `fetched_at`
- `attempts`
- `extractor_version`
- `extracted_title`
- `extracted_date`
- `language_detected`
- `body_sha1`
- `nc_word_count`
- `nc_country`
- `nc_paid_content`

`published_date`는 연구의 기준 날짜이므로 본문 추출기가 제공한 날짜로 덮어쓰지 않는다.


In [ ]:
EXTRACTION_COLUMNS = [
    "article_id",
    "url",
    "extractor_used",
    "extraction_status",
    "failure_reason",
    "body",
    "body_length",
    "http_status",
    "final_url",
    "fetched_at",
    "attempts",
    "extractor_version",
    "extracted_title",
    "extracted_date",
    "language_detected",
    "body_sha1",
    "nc_word_count",
    "nc_country",
    "nc_paid_content",
]


def empty_extraction_record():
    return {col: None for col in EXTRACTION_COLUMNS}


print("본문 추출 결과 컬럼 수:", len(EXTRACTION_COLUMNS))
print(EXTRACTION_COLUMNS)


## 13. 추출기 설치/선택은 벤치마크 후 결정

권장 비교 대상:

1. `trafilatura` — 범용 HTML 본문 추출기
2. `news-please` — 뉴스 기사 추출
3. `fundus` — 지원되는 publisher에 강점이 있는 뉴스 파서

중요한 점은 **Fundus를 무조건 첫 번째 추출기로 가정하지 않는 것**이다. 현재 데이터의 주요 도메인과 Fundus 지원 범위가 맞지 않을 수 있으므로 동일 표본에서 비교한다.

아래 셀은 선택적으로 패키지를 설치한다.


In [ ]:
# 필요할 때만 실행
# %pip install -q trafilatura news-please fundus


## 14. 벤치마크 실행 전 체크리스트

전체 추출 전에 `benchmark_sample`의 URL을 대상으로 각 추출기를 비교한다.

사람이 직접 확인할 때는 최소한 다음을 기록한다.

- 실제 기사 본문이 맞는가
- 본문이 충분히 긴가
- 메뉴/쿠키/에러 페이지가 섞였는가
- 제목과 본문이 맞는가
- 날짜가 맞는가
- 언어가 영어인가
- 같은 기사/신디케이션인가
- paywall 또는 차단 상태인가

추출 성공률만으로 기본 추출기를 결정하지 않는다. **본문 품질과 도메인별 성공률을 함께 평가한다.**

### 추출 상태 권장값

`success / empty / http_error / blocked / timeout / paywall_suspected / non_article / dead_link`

### 주의

- 4xx/dead link는 반복 요청하지 않는다.
- 5xx/timeout 등 일시 오류만 제한적으로 exponential backoff 재시도한다.
- paywall이나 Cloudflare 등의 제한을 우회하지 않는다.


## 15. 현재 단계 산출물 저장

이 셀은 새 분석용 메타데이터 파일을 만든다.

기존 `01_newscatcher_raw.csv`는 절대 덮어쓰지 않는다.

생성 파일:

- `05_article_index_v2.csv` — URL 중복 제거 후 전체 메타데이터
- `05_extraction_candidates_v2.csv` — 연구 기간 내 본문 추출 후보
- `05_extractor_benchmark_sample.csv` — 추출기 비교용 층화 표본

기존 `02_articles.csv`, `03_pre_diffbot_*`, `04_diagnostic_*`, `articles_partial.csv`와 이름을 섞지 않기 위해 `v2`를 붙인다.


In [ ]:
ARTICLE_INDEX_V2_FILE = DATA_DIR / "05_article_index_v2.csv"
EXTRACTION_CANDIDATES_V2_FILE = DATA_DIR / "05_extraction_candidates_v2.csv"
BENCHMARK_SAMPLE_FILE = DATA_DIR / "05_extractor_benchmark_sample.csv"

article_index.to_csv(
    ARTICLE_INDEX_V2_FILE,
    index=False,
    encoding="utf-8-sig",
)

extraction_candidates.to_csv(
    EXTRACTION_CANDIDATES_V2_FILE,
    index=False,
    encoding="utf-8-sig",
)

benchmark_sample.to_csv(
    BENCHMARK_SAMPLE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print("-", ARTICLE_INDEX_V2_FILE)
print("-", EXTRACTION_CANDIDATES_V2_FILE)
print("-", BENCHMARK_SAMPLE_FILE)


# 현재 단계 완료 기준

이 노트북을 끝까지 실행하면 다음 상태가 된다.

**NewsCatcher 원본**  
→ **URL 중복 제거**  
→ **원본 메타데이터 보존**  
→ **제목 중복은 삭제하지 않고 그룹화**  
→ **메타데이터 검토 플래그 생성**  
→ **연구 기간 내 본문 추출 후보 생성**  
→ **추출기 벤치마크 표본 생성**

그 다음 단계는 이 표본을 이용해 실제 본문 추출기를 비교하고, 기본 추출기 + fallback 조합을 확정한 뒤 전체 후보에 본문 추출을 적용하는 것이다.

## 절대 하지 않는 것

- NewsCatcher API 재수집
- 제목 중복을 `keep="first"`로 강제 삭제
- `hormuz_relevance_score == 0` 기사 일괄 삭제
- `description` 대신 잘린 `content`를 description으로 취급
- Diffbot Free 플랜으로 38만 건 이상 전체 본문 수집
- 본문 추출기의 날짜로 연구 기준 `published_date` 덮어쓰기
- paywall/Cloudflare/robots 제한 우회
